In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
!pip install biopython

In [ ]:
import os
import numpy as np
import pandas as pd
from pandas import Series, DataFrame
import random

import itertools
import math

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import TensorDataset, Dataset, DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, balanced_accuracy_score, matthews_corrcoef, precision_recall_curve
from tensorflow.keras.models import load_model

In [ ]:
new_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_combine/'
data_test_dir = os.path.join(new_model_dir, f'data/')
original_data_path = os.path.join(data_test_dir, 'all_data_with_SI.csv')
original_data_df = pd.read_csv(original_data_path)
original_data_df['SaSaRate']

In [ ]:
base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/'
new_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_combine/'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_combine/'
os.makedirs(save_dir, exist_ok=True)

all_folds_combined = []

for fold in range(2, 3):
    print(f"Processing fold {fold}...")

    # 1. Load the trained model
    model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
    model_path = os.path.join(model_dir, 'classification_model_with_val.h5')
    model = load_model(model_path)
    print(f"Model for fold {fold} loaded successfully.")

    data_test_dir = os.path.join(new_model_dir, f'data/')
    original_data_path = os.path.join(data_test_dir, 'all_data_with_SI.csv')
    original_data_df = pd.read_csv(original_data_path)
    merged_df = original_data_df[original_data_df['SaSaRate'] >= 0.2]
    #merged_df = original_data_df
    X_test = merged_df[[f'embedding_{i}' for i in range(0, 320)]].to_numpy()
    predictions = model.predict(X_test)
    merged_df['Predicted_Status'] = predictions

    # Fill missing predictions
    merged_df['Predicted_Status'] = pd.to_numeric(merged_df['Predicted_Status'], errors='coerce')

    # Normalize predictions within each protein group
    def normalize_group(x):
        if x.isnull().all() or (x.max() == x.min()):
            return np.nan
        else:
            return (x - x.min()) / (x.max() - x.min())

    merged_df['Normalized_Predicted_Status'] = merged_df.groupby('name')['Predicted_Status'].transform(normalize_group)

    all_folds_combined.append(merged_df)

combined_df = pd.concat(all_folds_combined, ignore_index=True)
combined_output_file = os.path.join(save_dir, 'all_folds_combined.csv')
combined_df.to_csv(combined_output_file, index=False)
print(f"All fold results combined and saved to {combined_output_file}.")

In [ ]:
combined_df = pd.read_csv(combined_output_file)
combined_df = combined_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['Normalized_Predicted_Status', 'Predicted_Status', 'Status'])

In [ ]:
combined_df = combined_df[combined_df['SaSaRate'] >= 0.6]

In [ ]:
combined_df

In [ ]:
combined_df = combined_df.dropna(subset=['Normalized_Predicted_Status'])
combined_df = combined_df.dropna(subset=['Predicted_Status'])
combined_df = combined_df.dropna(subset=['Status'])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set up the matplotlib figure
plt.figure(figsize=(10, 6))

# Plot the distribution of pred values for each Status
sns.histplot(data=combined_df, x='Normalized_Predicted_Status', hue='Status', element='step', stat="density", common_norm=False)

# Provide labels and title
plt.xlabel('Prediction Value')
plt.ylabel('Density')
plt.title('Distribution of Prediction Values by Status')
plt.legend(title='Status')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt

y_true = combined_df['Status'].replace({'Interacting': 1, 'Non-Interacting': 0}).astype(int)
#y_pred = combined_df.index.isin(combined_df['Predicted_Status'].nlargest(700).index).astype(int)
y_pred = (combined_df['Predicted_Status'] >= 0.5).astype(int)
# y_scores = combined_df.index.isin(combined_df['Normalized_Predicted_Status'].nlargest(700).index).astype(int)
y_scores = combined_df['Normalized_Predicted_Status']

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_true, y_scores)

print("Classification Report:")
print(classification_report(y_true, y_pred))
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC: {auc:.4f}")

conf_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 6))
plt.imshow(conf_matrix, cmap="Blues", interpolation="nearest")
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], ['Non-Interacting', 'Interacting'])
plt.yticks([0, 1], ['Non-Interacting', 'Interacting'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

for i in range(2):
    for j in range(2):
        plt.text(j, i, conf_matrix[i, j], ha="center", va="center", color="red")

plt.show()

In [ ]:
from sklearn.metrics import (
    roc_curve,
    auc as compute_auc,
    balanced_accuracy_score,
    matthews_corrcoef,
    precision_recall_curve
)

# Compute ROC curve and ROC-AUC
fpr, tpr, _ = roc_curve(y_true, y_pred)
roc_auc_score = compute_auc(fpr, tpr)
print(f'ROC-AUC: {roc_auc_score:.2f}')

# Compute Balanced Accuracy (BAC)
y_pred_binary = (combined_df['Normalized_Predicted_Status'] >= 0.5).astype(int) #combined_df.index.isin(combined_df['Normalized_Predicted_Status'].nlargest(700).index).astype(int)
balanced_acc = balanced_accuracy_score(y_true, y_pred_binary)
print(f'BAC: {balanced_acc:.2f}')

# Compute Matthews Correlation Coefficient (MCC)
mcc_score = matthews_corrcoef(y_true, y_pred_binary)
print(f'MCC: {mcc_score:.2f}')

# Compute Precision-Recall curve and PR-AUC
precision, recall, _ = precision_recall_curve(y_true, y_pred_binary)
pr_auc_score = compute_auc(recall, precision)
print(f'PR-AUC: {pr_auc_score:.2f}')

# Compute Nantigens (number of unique antigens)
num_unique_antigens = combined_df['name'].nunique()
print(f'Nantigens: {num_unique_antigens}')

# Compute Fpredicted (mean fraction of surface residues predicted as epitopes)
fraction_predicted = np.mean(y_pred_binary) * 100
print(f'Fpredicted: {fraction_predicted:.2f}%')


In [ ]:
import os
import pandas as pd
from Bio.PDB import PDBParser, PDBIO, is_aa

# ---------- 可调参数 ----------
BEST_NAMES   = ["6vyb"]          # 需要处理的 pdb id
THRESHOLD    = 0.90              # 预测值 ≥ THRESHOLD 记为 1.0，否则 0.0
INPUT_PDB_DIR  = "/content/drive/MyDrive/@antigen_epitope/abdb_combine/data/structures/"
OUTPUT_PDB_DIR = "/content/drive/MyDrive/@antigen_epitope/abdb_combine/"

best_df = combined_df[combined_df['name'].isin(BEST_NAMES)].copy()
best_df['ResID'] = best_df['ResID'].astype(str)
status_dict = { (row['Chain'], row['ResID']) : row['Normalized_Predicted_Status']
                for _, row in best_df.iterrows() }
parser = PDBParser(QUIET=True)
io     = PDBIO()

for pdb_name in best_df['name'].unique():
    pdb_file = os.path.join(INPUT_PDB_DIR, f"{pdb_name}.pdb")
    structure = parser.get_structure(pdb_name, pdb_file)

    for model in structure:
        for chain in model:
            for residue in chain:
                if not is_aa(residue, standard=True):
                    continue

                # ---- 提取当前残基的 (Chain, ResID) ----
                res_id_tuple = residue.get_id()           # (' ', 123, ' ')
                res_id_str   = str(res_id_tuple[1])       # 仅取数字部分即可
                key = (chain.id, res_id_str)

                # ---- 判定 B‑factor 值 ----
                pred_val = status_dict.get(key, 0.0)
                new_b = 1.0 if pred_val >= THRESHOLD else 0.0

                for atom in residue.get_atoms():
                    atom.set_bfactor(new_b)

    # ---------- 保存 ----------
    os.makedirs(OUTPUT_PDB_DIR, exist_ok=True)
    out_path = os.path.join(OUTPUT_PDB_DIR, f"{pdb_name}.pdb")
    io.set_structure(structure)
    io.save(out_path)
    print(f"[OK] 处理并保存：{out_path}")


In [ ]:
import os
import pandas as pd
from Bio.PDB import PDBParser, PDBIO, is_aa

# ---------- 可调参数 ----------
BEST_NAMES   = ["6vyb"]          # 需要处理的 pdb id
THRESHOLD    = 0.90              # 预测值 ≥ THRESHOLD 记为 1.0，否则 0.0
INPUT_PDB_DIR  = "/content/drive/MyDrive/@antigen_epitope/abdb_combine/data/structures/"
OUTPUT_PDB_DIR = "/content/drive/MyDrive/@antigen_epitope/abdb_combine/"

best_df = combined_df[combined_df['name'].isin(BEST_NAMES)].copy()
best_df['ResID'] = best_df['ResID'].astype(str)
status_dict = { (row['Chain'], row['ResID']) : row['Status']
                for _, row in best_df.iterrows() }
parser = PDBParser(QUIET=True)
io     = PDBIO()

for pdb_name in best_df['name'].unique():
    pdb_file = os.path.join(INPUT_PDB_DIR, f"{pdb_name}.pdb")
    structure = parser.get_structure(pdb_name, pdb_file)

    for model in structure:
        for chain in model:
            for residue in chain:
                if not is_aa(residue, standard=True):
                    continue

                # ---- 提取当前残基的 (Chain, ResID) ----
                res_id_tuple = residue.get_id()           # (' ', 123, ' ')
                res_id_str   = str(res_id_tuple[1])       # 仅取数字部分即可
                key = (chain.id, res_id_str)

                # ---- 判定 B‑factor 值 ----
                pred_val = status_dict.get(key, 0.0)
                new_b = 1.0 if pred_val == 'Interacting' else 0.0

                for atom in residue.get_atoms():
                    atom.set_bfactor(new_b)

    # ---------- 保存 ----------
    os.makedirs(OUTPUT_PDB_DIR, exist_ok=True)
    out_path = os.path.join(OUTPUT_PDB_DIR, f"{pdb_name}.pdb")
    io.set_structure(structure)
    io.save(out_path)
    print(f"[OK] 处理并保存：{out_path}")